In [1]:
import json
from collections import defaultdict
from pathlib import Path

import equinox as eqx
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np
import wandb
import xarray as xr
from context_flux_no.metrics import relative_L2_error, relative_L_infty_error
from context_flux_no.training.io import load_model
from einops import rearrange
from jaxtyping import Array, Float
from matplotlib.animation import ArtistAnimation
from tqdm import tqdm


datadir = Path("../../data")
checkpoint_dir = Path("../../checkpoints/cubicflux_1d")

jax.config.update("jax_default_device", jax.devices("gpu")[2])

In [2]:
@eqx.filter_jit
def compute_metrics(model, u, args, context_length=20):
    context, u_data = u[:context_length], u[context_length:]

    u_pred = model.rollout(context, args, num_steps=len(u_data))[0]

    return {
        "l2_onestep": relative_L2_error(u_pred[0], u_data[0]),
        "l_inf_onestep": relative_L_infty_error(u_pred[0], u_data[0]),
        "l2_rollout": relative_L2_error(u_pred, u_data),
        "l_inf_rollout": relative_L_infty_error(u_pred, u_data),
    }

In [6]:
model_paths = {
    5: [
        # "seed=0/",
        "seed=10/26-07-25-03_23_49",
        "seed=20/26-07-25-14_07_23",
    ],
    10: [
        # "seed=0/26-04-16-00_31_47",
        "seed=10/26-07-25-07_25_05",
        "seed=20/26-07-25-15_45_14",
    ],
    20: [
        "seed=0/26-04-16-00_31_47",
        "seed=10/26-04-17-07_57_49",
        "seed=20/26-04-17-10_50_08",
    ],
    40: [
        # "seed=0/26-04-16-00_31_47",
        "seed=10/26-07-25-10_56_49",
        "seed=20/26-07-25-16_04_56",
    ],
}

In [7]:
dataset_test = (
    xr.open_dataset(
        datadir
        / "datasets/cubic_no_source/data/test/cubic_no_source_large_test_seed=10.hdf5",
        engine="h5netcdf",
        chunks={},
    )
    .isel(t=slice(None, None, 10))
    .isel({"t": slice(0, 99)})
)
dt = float(dataset_test["t"][1] - dataset_test["t"][0])
dx = float(dataset_test["x"][1] - dataset_test["x"][0])

values = rearrange(dataset_test["values"].values, "pde ic ... -> (pde ic) ...")
segments = np.lib.stride_tricks.sliding_window_view(values, 60, axis=1)
segments = rearrange(segments, "batch t0 c x t -> (batch t0) t c x")
segments.shape

(400000, 60, 1, 100)

In [15]:
dt = float(dataset_test["t"][1] - dataset_test["t"][0])
dx = float(dataset_test["x"][1] - dataset_test["x"][0])
results_dict = defaultdict(list)
for ctx_length, paths in model_paths.items():
    for p in tqdm(paths):
        model = load_model(checkpoint_dir / "HyperFluxFNOLocal/OneStepLoss" / p)
        model = eqx.nn.inference_mode(model, True)
        results = jax.lax.map(
            eqx.filter_jit(
                lambda u: compute_metrics(model, u, (dt, dx), context_length=ctx_length)
            ),
            segments[:, : (ctx_length + 20)],
            batch_size=6000,
        )
        print(jax.tree.map(lambda x: x.shape, results))
        results_dict[ctx_length].append(jax.tree.map(jnp.mean, results))

  0%|          | 0/2 [00:00<?, ?it/s]/home/jhko725/projects/CONTEXT_FLUX_NO/src/context_flux_no/models/multiphysics/hyperfluxfno/utils.py:53: UserWarning: TRecViTEncoder supports variable in_timesteps. The given 
                    in_timesteps value will be ignored.
  warnings.warn(
/home/jhko725/projects/CONTEXT_FLUX_NO/src/context_flux_no/nn/structured_linear.py:40: UserWarning: out_features is not divisible by num_blocks. Output vector 
            will be truncated to the requested size.
  warnings.warn("""out_features is not divisible by num_blocks. Output vector


(5, 2, 100)


E0731 12:34:48.236533 3147617 cuda_timer.cc:87] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
E0731 12:34:48.494538 3147658 cuda_timer.cc:87] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
 50%|█████     | 1/2 [01:08<01:08, 68.18s/it]

{'l2_onestep': (400000,), 'l2_rollout': (400000,), 'l_inf_onestep': (400000,), 'l_inf_rollout': (400000,)}


100%|██████████| 2/2 [02:12<00:00, 66.32s/it]


...

(40, 2, 100)


E0731 12:53:37.484689 3147662 cuda_timer.cc:87] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
 50%|█████     | 1/2 [06:07<06:07, 367.00s/it]

{'l2_onestep': (400000,), 'l2_rollout': (400000,), 'l_inf_onestep': (400000,), 'l_inf_rollout': (400000,)}


100%|██████████| 2/2 [12:05<00:00, 362.81s/it]

{'l2_onestep': (400000,), 'l2_rollout': (400000,), 'l_inf_onestep': (400000,), 'l_inf_rollout': (400000,)}


In [16]:
results_dict

defaultdict(list,
            {5: [{'l2_onestep': Array(0.00645192, dtype=float32),
               'l2_rollout': Array(0.10854306, dtype=float32),
               'l_inf_onestep': Array(0.02256064, dtype=float32),
               'l_inf_rollout': Array(0.63115954, dtype=float32)},
              {'l2_onestep': Array(0.00648508, dtype=float32),
               'l2_rollout': Array(0.11115244, dtype=float32),
               'l_inf_onestep': Array(0.02299029, dtype=float32),
               'l_inf_rollout': Array(0.64394474, dtype=float32)}],
             10: [{'l2_onestep': Array(0.00532477, dtype=float32),
               'l2_rollout': Array(0.07459842, dtype=float32),
               'l_inf_onestep': Array(0.01914159, dtype=float32),
               'l_inf_rollout': Array(0.4894069, dtype=float32)},
              {'l2_onestep': Array(0.00533847, dtype=float32),
               'l2_rollout': Array(0.07721397, dtype=float32),
               'l_inf_onestep': Array(0.01915929, dtype=float32),
      

In [17]:
for ctx_length, res in results_dict.items():
    if ctx_length == 20:
        res_ = jax.tree.transpose(jax.tree.structure(["*"] * 3), None, res)
    else:
        res_ = jax.tree.transpose(jax.tree.structure(["*"] * 2), None, res)
    stats = jax.tree.map(
        lambda list_: {
            "mean": jnp.mean(jnp.asarray(list_)),
            "std": jnp.std(jnp.asarray(list_)),
        },
        res_,
        is_leaf=lambda x: isinstance(x, list),
    )
    for k, v in stats.items():
        print(ctx_length, f"{k}: {v['mean']:.2e}" + "±" + f"{v['std']:.2e}")

5 l2_onestep: 6.47e-03±1.66e-05
5 l2_rollout: 1.10e-01±1.30e-03
5 l_inf_onestep: 2.28e-02±2.15e-04
5 l_inf_rollout: 6.38e-01±6.39e-03
10 l2_onestep: 5.33e-03±6.85e-06
10 l2_rollout: 7.59e-02±1.31e-03
10 l_inf_onestep: 1.92e-02±8.85e-06
10 l_inf_rollout: 4.96e-01±6.14e-03
20 l2_onestep: 4.37e-03±1.20e-04
20 l2_rollout: 5.42e-02±2.23e-03
20 l_inf_onestep: 1.63e-02±6.42e-04
20 l_inf_rollout: 3.78e-01±1.61e-02
40 l2_onestep: 3.39e-03±9.22e-05
40 l2_rollout: 3.88e-02±7.34e-04
40 l_inf_onestep: 1.30e-02±3.41e-04
40 l_inf_rollout: 2.78e-01±3.68e-03
